<a href="https://colab.research.google.com/github/plv02b107/earnings-volatility-crush/blob/main/MWPL_Causality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MWPL Causality Analysis
Research Question:
Does Market Wide Position Limit (MWPL) contain predictive information for subsequent stock returns?

In [ ]:
!pip install linearmodels -q

import io
import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm

from google.colab import files
from scipy.stats import ttest_1samp, ttest_ind, combine_pvalues, chi2
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.stats.multitest import multipletests
from linearmodels.panel import PanelOLS

warnings.filterwarnings("ignore")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 4.1 MB/s eta 0:00:00


# 1. CONFIGURATION

In [ ]:
DATE_COL   = "Date"
STOCK_COL  = "NSE Symbol"
PRICE_COL  = "Close"
MWPL_COL   = "MWPL_%"
OI_COL     = "OI_Change_%"
VOL_COL    = "Vol_Percentile"

MIN_OBS = 30
MAX_LAG = 3
TCOST = 0.002

# 2. LOAD DATA

In [ ]:
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(io.BytesIO(uploaded[file_name]))

print("File uploaded:", file_name)
print("Raw shape:", df.shape)

Saving 01_final_merged_mwpl_price_dataset.csv to 01_final_merged_mwpl_price_dataset.csv
File uploaded: 01_final_merged_mwpl_price_dataset.csv
Raw shape: (12914, 35)


# 3. BASIC VALIDATION AND CLEANING

In [ ]:
required_cols = [DATE_COL, STOCK_COL, PRICE_COL, MWPL_COL, OI_COL]


df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

for col in [PRICE_COL, MWPL_COL, OI_COL, VOL_COL]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df = (
    df.dropna(subset=[DATE_COL, STOCK_COL, PRICE_COL, MWPL_COL, OI_COL])
      .sort_values([STOCK_COL, DATE_COL])
      .reset_index(drop=True)
)
has_vol = VOL_COL in df.columns

print("\nData audit")
print("Rows:", len(df))
print("Stocks:", df[STOCK_COL].nunique())
print("Dates:", df[DATE_COL].nunique())
print("Date range:", df[DATE_COL].min().date(), "to", df[DATE_COL].max().date())



Data audit
Rows: 12694
Stocks: 215
Dates: 61
Date range: 2026-02-03 to 2026-05-06


# 4. FEATURE ENGINEERING

In [ ]:
g = df.groupby(STOCK_COL, sort=False)

df["Return_1D"] = np.log(df[PRICE_COL] / g[PRICE_COL].shift(1))
df["Return_t1"] = g["Return_1D"].shift(-1)

df["Return_Fwd_5D"] = np.log(g[PRICE_COL].shift(-5) / df[PRICE_COL])
df["Return_Past_3D"] = np.log(df[PRICE_COL] / g[PRICE_COL].shift(3))

df["MWPL_diff"] = g[MWPL_COL].diff()
df["MWPL_diff_lag1"] = g["MWPL_diff"].shift(1)

df["OI_Change_lag1"] = g[OI_COL].shift(1)

for th in [70, 80, 85, 90, 95]:
    df[f"High_MWPL_{th}"] = (df[MWPL_COL] >= th).astype(int)

df["At_Ban"] = (df[MWPL_COL] >= 95).astype(int)
df["MWPL_c"] = df[MWPL_COL] - 95

df["Alpha_t1"] = df["Return_t1"]


# 5. FINAL MODEL SAMPLE

In [ ]:
model_cols = [
    "Alpha_t1",
    "Return_t1",
    "Return_Fwd_5D",
    "Return_Past_3D",
    "MWPL_diff",
    "MWPL_diff_lag1",
    "OI_Change_lag1",
    MWPL_COL,
    OI_COL
]
if has_vol:
    model_cols.append(VOL_COL)

clean = df.dropna(subset=model_cols).copy()

valid_stocks = clean.groupby(STOCK_COL).size()
valid_stocks = valid_stocks[valid_stocks >= MIN_OBS].index

clean = clean[clean[STOCK_COL].isin(valid_stocks)].copy()
print("\nFinal sample")
print("Rows:", len(clean))
print("Stocks:", clean[STOCK_COL].nunique())
print("Trading days:", clean[DATE_COL].nunique())




Final sample
Rows: 10620
Stocks: 205
Trading days: 52


# 6. STATIONARITY CHECK

In [ ]:
def adf_rejection_rate(data, col):
    results = []

    for _, temp in data.groupby(STOCK_COL):
        x = temp[col].dropna()

        if len(x) >= MIN_OBS:
            try:
                pval = adfuller(x, autolag="AIC")[1]
                results.append(pval < 0.05)
            except:
                pass

    return np.mean(results) if results else np.nan
stationarity_table = pd.DataFrame({
    "Variable": [MWPL_COL, "MWPL_diff", "Return_t1", "Alpha_t1", OI_COL],
    "ADF rejection share": [
        adf_rejection_rate(clean, MWPL_COL),
        adf_rejection_rate(clean, "MWPL_diff"),
        adf_rejection_rate(clean, "Return_t1"),
        adf_rejection_rate(clean, "Alpha_t1"),
        adf_rejection_rate(clean, OI_COL)
    ]
})

print("\nStationarity check")
display(stationarity_table)



Stationarity check


,Variable,ADF rejection share
0,MWPL_%,0.073171
1,MWPL_diff,0.921951
2,Return_t1,0.882927
3,Alpha_t1,0.882927
4,OI_Change_%,0.995122


# 7. MAIN PANEL MODEL

*   Two-way fixed effects
*   Stock FE controls stock-specific differences.
*   Date FE controls market-wide shocks.






In [ ]:
panel = clean.set_index([STOCK_COL, DATE_COL])

x_vars = ["MWPL_diff_lag1", "OI_Change_lag1", "Return_Past_3D"]

if has_vol:
    x_vars.append(VOL_COL)

X = sm.add_constant(panel[x_vars])
y = panel["Alpha_t1"]

panel_model = PanelOLS(
    y,
    X,
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
panel_res = panel_model.fit(
    cov_type="kernel",
    kernel="newey-west",
    bandwidth=3
)

print("\nMain Panel Regression")
print(panel_res.summary)


panel_summary = pd.DataFrame({
    "Variable": x_vars,
    "Coefficient": [panel_res.params.get(v, np.nan) for v in x_vars],
    "P-value": [panel_res.pvalues.get(v, np.nan) for v in x_vars]
})
panel_summary["Significance"] = np.select(
    [
        panel_summary["P-value"] < 0.01,
        panel_summary["P-value"] < 0.05,
        panel_summary["P-value"] < 0.10
    ],
    ["***", "**", "*"],
    default=""
)

print("\nPanel result summary")
display(panel_summary)


Main Panel Regression
                          PanelOLS Estimation Summary                           
Dep. Variable:               Alpha_t1   R-squared:                        0.0009
Estimator:                   PanelOLS   R-squared (Between):             -0.0508
No. Observations:               10620   R-squared (Within):               0.0023
Date:                Fri, May 08 2026   R-squared (Overall):              0.0018
Time:                        14:15:05   Log-likelihood                 2.847e+04
Cov. Estimator:        Driscoll-Kraay                                           
                                        F-statistic:                      2.3889
Entities:                         205   P-value                           0.0487
Avg Obs:                       51.805   Distribution:                 F(4,10360)
Min Obs:                       33.000                                           
Max Obs:                       52.000   F-statistic (robust):             1.3618
     

,Variable,Coefficient,P-value,Significance
0,MWPL_diff_lag1,-0.000113,0.303461,
1,OI_Change_lag1,-0.000020,0.276277,
2,Return_Past_3D,-0.008690,0.388859,
3,Vol_Percentile,0.001443,0.210083,


## 8. ENTITY-LEVEL GRANGER CAUSALITY
 -Direction tested

1.   MWPL_diff  -> Alpha_t1
2.   Alpha_t1   -> MWPL_diff



In [ ]:
def effective_n(data, variable):
    try:
        pivot = data.pivot(index=DATE_COL, columns=STOCK_COL, values=variable)
        corr = pivot.corr().fillna(0).values
        eig = np.linalg.eigvalsh(corr)
        eig = np.maximum(eig, 0)

        meff = (eig.sum() ** 2) / (eig ** 2).sum()
        return max(1, int(round(meff)))

    except:
        return data[STOCK_COL].nunique()


def run_entity_granger(data, target, cause, maxlag=MAX_LAG):
    rows = []

    for stock, temp in data.groupby(STOCK_COL):
        temp = temp.sort_values(DATE_COL)
        test_data = temp[[target, cause]].dropna()

        if len(test_data) < MIN_OBS + maxlag + 1:
            continue

        try:
            result = grangercausalitytests(
                test_data[[target, cause]],
                maxlag=maxlag,
                verbose=False
            )

            pvals = {
                lag: result[lag][0]["ssr_ftest"][1]
                for lag in range(1, maxlag + 1)
            }

            best_lag = min(pvals, key=pvals.get)

            rows.append({
                STOCK_COL: stock,
                "n_obs": len(test_data),
                "best_lag": best_lag,
                "min_pvalue": pvals[best_lag],
                **{f"p_lag{lag}": pvals[lag] for lag in range(1, maxlag + 1)}
            })

        except:
            continue

    out = pd.DataFrame(rows)

    if out.empty:
        return out, {}

    reject, p_bh, _, _ = multipletests(
        out["min_pvalue"],
        alpha=0.05,
        method="fdr_bh"
    )

    out["p_bh"] = p_bh
    out["reject_bh"] = reject

    meff = effective_n(data, cause)

    fisher_stat, fisher_p_naive = combine_pvalues(out["min_pvalue"])
    fisher_p_adj = chi2.sf(fisher_stat, df=2 * meff)

    summary = {
        "tested_stocks": len(out),
        "effective_N": meff,
        "raw_share_p<0.05": (out["min_pvalue"] < 0.05).mean(),
        "BH_reject_share": out["reject_bh"].mean(),
        "fisher_p_adj": fisher_p_adj
    }

    return out.sort_values("min_pvalue"), summary


forward_granger, forward_summary = run_entity_granger(
    clean,
    target="Alpha_t1",
    cause="MWPL_diff"
)

reverse_granger, reverse_summary = run_entity_granger(
    clean,
    target="MWPL_diff",
    cause="Alpha_t1"
)

granger_comparison = pd.DataFrame([
    {"Direction": "MWPL_diff -> Alpha_t1", **forward_summary},
    {"Direction": "Alpha_t1 -> MWPL_diff", **reverse_summary}
])

print("\nGranger causality comparison")
display(granger_comparison)

print("\nTop forward Granger stocks")
display(forward_granger.head(15))

print("\nTop reverse Granger stocks")
display(reverse_granger.head(15))






Granger causality comparison


,Direction,tested_stocks,effective_N,raw_share_p<0.05,BH_reject_share,fisher_p_adj
0,MWPL_diff -> Alpha_t1,204,17,0.068627,0.004902,4.210285e-105
1,Alpha_t1 -> MWPL_diff,204,4,0.436275,0.289216,6.219117e-294



Top forward Granger stocks


,NSE Symbol,n_obs,best_lag,min_pvalue,p_lag1,p_lag2,p_lag3,p_bh,reject_bh
4,ADANIENT,52,3,0.000051,0.050730,0.224225,0.000051,0.010487,True
5,ADANIGREEN,52,3,0.000759,0.002087,0.037245,0.000759,0.077469,False
18,AXISBANK,52,3,0.005299,0.060822,0.197789,0.005299,0.360358,False
43,COALINDIA,52,1,0.009840,0.009840,0.030039,0.067879,0.432477,False
113,LTF,52,1,0.010816,0.010816,0.105420,0.068819,0.432477,False
72,HDFCBANK,52,3,0.012720,0.592815,0.519654,0.012720,0.432477,False
106,KOTAKBANK,52,3,0.022501,0.870825,0.981414,0.022501,0.494368,False
197,VBL,52,3,0.027608,0.168971,0.103147,0.027608,0.494368,False
13,ASHOKLEY,52,3,0.035614,0.516809,0.205103,0.035614,0.494368,False
66,GODREJPROP,52,1,0.041498,0.041498,0.133555,0.094576,0.494368,False



Top reverse Granger stocks


,NSE Symbol,n_obs,best_lag,min_pvalue,p_lag1,p_lag2,p_lag3,p_bh,reject_bh
84,IDFCFIRSTB,52,1,4.525181e-08,4.525181e-08,4.703582e-07,0.000005,0.000009,True
159,RBLBANK,52,1,4.253489e-07,4.253489e-07,5.153768e-06,0.000001,0.000043,True
133,NTPC,52,1,2.601466e-06,2.601466e-06,1.341097e-05,0.000062,0.000127,True
1,ABB,52,1,2.702437e-06,2.702437e-06,9.904867e-06,0.000011,0.000127,True
76,HINDPETRO,52,1,3.122927e-06,3.122927e-06,2.817893e-05,0.000094,0.000127,True
44,COFORGE,52,2,4.924553e-06,8.186281e-04,4.924553e-06,0.000033,0.000167,True
145,PFC,52,1,1.204880e-05,1.204880e-05,1.654242e-05,0.000098,0.000348,True
196,UPL,52,1,1.365925e-05,1.365925e-05,8.406264e-05,0.000429,0.000348,True
85,IEX,52,1,1.952965e-05,1.952965e-05,1.625956e-04,0.000783,0.000443,True
67,GRASIM,52,2,2.243585e-05,4.559632e-01,2.243585e-05,0.000058,0.000458,True


# 9. HIGH MWPL EVENT STUDY

In [ ]:
event_rows = []

for th in [70, 80, 85, 90, 95]:
    s1 = clean.loc[clean[MWPL_COL] >= th, "Alpha_t1"].dropna()
    s5 = clean.loc[clean[MWPL_COL] >= th, "Return_Fwd_5D"].dropna()

    p1 = ttest_1samp(s1, 0)[1] if len(s1) > 1 else np.nan
    p5 = ttest_1samp(s5, 0)[1] if len(s5) > 1 else np.nan

    event_rows.append({
        "MWPL threshold": th,
        "Obs 1D": len(s1),
        "Mean next-day alpha": s1.mean(),
        "P-value 1D": p1,
        "Obs 5D": len(s5),
        "Mean 5D return": s5.mean(),
        "P-value 5D": p5
    })

event_table = pd.DataFrame(event_rows)
print("\nHigh MWPL event study")
display(event_table)




High MWPL event study


,MWPL threshold,Obs 1D,Mean next-day alpha,P-value 1D,Obs 5D,Mean 5D return,P-value 5D
0,70,310,-0.000359,0.798632,310,0.002156,0.424616
1,80,137,-0.000127,0.953769,137,0.004315,0.267849
2,85,94,-0.000127,0.960169,94,0.008153,0.089381
3,90,44,-0.000577,0.868031,44,0.001971,0.781175
4,95,12,0.001446,0.837212,12,0.011091,0.502822


# 10. CROSS-SECTIONAL LONG-SHORT TEST
Top 10% MWPL stocks vs Bottom 10% MWPL stocks each day

In [ ]:
ranked = clean.copy()
ranked["MWPL_rank"] = ranked.groupby(DATE_COL)[MWPL_COL].rank(pct=True)

top = ranked.loc[ranked["MWPL_rank"] >= 0.90, "Alpha_t1"].dropna()
bottom = ranked.loc[ranked["MWPL_rank"] <= 0.10, "Alpha_t1"].dropna()

spread = top.mean() - bottom.mean()
p_spread = ttest_ind(top, bottom, equal_var=False)[1]

long_short_table = pd.DataFrame({
    "Group": ["Top 10% MWPL", "Bottom 10% MWPL", "Long-short spread"],
    "Mean Alpha": [top.mean(), bottom.mean(), spread],
    "Median Alpha": [top.median(), bottom.median(), np.nan],
    "N": [len(top), len(bottom), np.nan],
    "P-value": [np.nan, np.nan, p_spread]
})

print("\nCross-sectional long-short test")
display(long_short_table)


Cross-sectional long-short test


,Group,Mean Alpha,Median Alpha,N,P-value
0,Top 10% MWPL,0.000409,0.000748,1092.0,NaN
1,Bottom 10% MWPL,-0.000762,0.000024,1040.0,NaN
2,Long-short spread,0.001171,NaN,NaN,0.24013


# 11. WALK-FORWARD TEST WITH TRANSACTION COST

In [ ]:
split_date = clean[DATE_COL].quantile(0.75)

train = clean[clean[DATE_COL] <= split_date].copy()
test = clean[clean[DATE_COL] > split_date].copy()


def signal_test(data, threshold):
    r = data.loc[data[MWPL_COL] >= threshold, "Alpha_t1"].dropna()

    if len(r) <= 1:
        return {
            "Threshold": threshold,
            "Trades": len(r),
            "Mean gross": np.nan,
            "Mean net": np.nan,
            "Positive net share": np.nan,
            "P-value": np.nan
        }

    net = r - TCOST
    return {
        "Threshold": threshold,
        "Trades": len(r),
        "Mean gross": r.mean(),
        "Mean net": net.mean(),
        "Positive net share": (net > 0).mean(),
        "P-value": ttest_1samp(net, 0)[1]
    }


wf_rows = []

for th in [70, 80, 85, 90, 95]:
    train_row = signal_test(train, th)
    train_row["Sample"] = "Train"
    test_row = signal_test(test, th)
    test_row["Sample"] = "Test"

    wf_rows.extend([train_row, test_row])

walk_forward_table = pd.DataFrame(wf_rows)

print("\nWalk-forward test after transaction cost")
display(walk_forward_table)




Walk-forward test after transaction cost


,Threshold,Trades,Mean gross,Mean net,Positive net share,P-value,Sample
0,70,266,-0.001059,-0.003059,0.462406,0.055396,Train
1,70,44,0.003871,0.001871,0.522727,0.434094,Test
2,80,112,-0.000759,-0.002759,0.464286,0.292744,Train
3,80,25,0.002702,0.000702,0.480000,0.793334,Test
4,85,79,-0.001567,-0.003567,0.468354,0.228273,Train
5,85,15,0.007456,0.005456,0.600000,0.121475,Test
6,90,35,-0.002930,-0.004930,0.400000,0.246619,Train
7,90,9,0.008571,0.006571,0.666667,0.084440,Test
8,95,8,-0.003785,-0.005785,0.375000,0.562183,Train
9,95,4,0.011910,0.009910,0.750000,0.228662,Test


# 12. REGRESSION DISCONTINUITY AROUND 95% MWPL

In [ ]:
bandwidth = 5

rd = clean[
    (clean[MWPL_COL] >= 95 - bandwidth) &
    (clean[MWPL_COL] <= 95 + bandwidth)
].copy()

print("\nRD sample around 95% threshold")
print("Observations:", len(rd))
print("Below 95:", (rd[MWPL_COL] < 95).sum())
print("At/above 95:", (rd[MWPL_COL] >= 95).sum())

if len(rd) >= 30:
    rd["D"] = (rd[MWPL_COL] >= 95).astype(int)
    rd["MWPL_c"] = rd[MWPL_COL] - 95
    rd["D_x_MWPL_c"] = rd["D"] * rd["MWPL_c"]

    X_rd = sm.add_constant(rd[["D", "MWPL_c", "D_x_MWPL_c"]])
    y_rd = rd["Alpha_t1"]
    rd_res = sm.OLS(y_rd, X_rd).fit(cov_type="HC3")

    print("\nRegression discontinuity result")
    print(rd_res.summary())

else:
    print("Not enough observations near the 95% threshold for RD estimation.")





RD sample around 95% threshold
Observations: 43
Below 95: 32
At/above 95: 11

Regression discontinuity result
                            OLS Regression Results                            
Dep. Variable:               Alpha_t1   R-squared:                       0.103
Model:                            OLS   Adj. R-squared:                  0.034
Method:                 Least Squares   F-statistic:                     1.450
Date:                Fri, 08 May 2026   Prob (F-statistic):              0.243
Time:                        14:15:15   Log-Likelihood:                 104.42
No. Observations:                  43   AIC:                            -200.8
Df Residuals:                      39   BIC:                            -193.8
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
--------------------

# Key Takeaways:

1.No evidence of stock-level alpha
MWPL_diff_lag1 is not statistically significant after
two-way fixed effects and Newey-West standard errors.

2.Reverse causality dominates
Returns -> MWPL is much stronger than MWPL -> Returns,
suggesting MWPL reflects positioning, not prediction.

3.Extreme MWPL levels are underpowered
Very few observations near 90-95%, so event study and
RD results should be treated as exploratory.

4.No tradeable signal after costs
Walk-forward results do not consistently generate
positive net returns.

# Conclusion:
MWPL does not provide a robust predictive signal for
next-day returns in this sample. It behaves more as a
crowding/positioning indicator than a source of alpha.